# AI_Agent_Workshop_Day2_03 — Evaluate and Pipeline

In this notebook we evaluate the service assistant and connect the workflow to a lightweight DVC pipeline.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "eval").exists():
    ROOT = Path("day2_workshop")

EVAL_DIR = ROOT / "eval"
DATA_DIR = ROOT / "data"
ARTIFACTS_DIR = ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

eval_df = pd.read_csv(EVAL_DIR / "service_eval_set.csv")
catalog = pd.read_csv(DATA_DIR / "service_catalog.csv")
eval_df

## Evaluation criteria

We will track a small set of task-focused metrics:

- jurisdiction classification accuracy,
- responsible-body match,
- source presence,
- next-step usefulness,
- abstention quality for ambiguous cases.

In [ ]:
import re

def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())

def keyword_baseline_predict(question: str, df: pd.DataFrame) -> dict:
    q_tokens = set(tokenize(question))
    best_score = -1
    best_row = None
    for _, row in df.iterrows():
        text = " ".join([
            str(row["service_name"]),
            str(row["description"]),
            str(row["keywords"]),
            str(row["responsible_body"]),
        ]).lower()
        score = sum(tok in tokenize(text) for tok in q_tokens)
        if score > best_score:
            best_score = score
            best_row = row
    return {
        "predicted_service_name": best_row["service_name"],
        "predicted_jurisdiction_level": best_row["jurisdiction_level"],
        "predicted_responsible_body": best_row["responsible_body"],
        "sources": [best_row["source_url"]],
    }

In [ ]:
predictions = []
for _, row in eval_df.iterrows():
    pred = keyword_baseline_predict(row["question"], catalog)
    pred["question"] = row["question"]
    pred["expected_jurisdiction_level"] = row["expected_jurisdiction_level"]
    pred["expected_responsible_body"] = row["expected_responsible_body"]
    predictions.append(pred)

pred_df = pd.DataFrame(predictions)
pred_df

In [ ]:
metrics = {
    "jurisdiction_accuracy": float(
        (pred_df["predicted_jurisdiction_level"] == pred_df["expected_jurisdiction_level"]).mean()
    ),
    "responsible_body_accuracy": float(
        (pred_df["predicted_responsible_body"] == pred_df["expected_responsible_body"]).mean()
    ),
    "source_presence_rate": float(
        pred_df["sources"].apply(lambda x: len(x) > 0).mean()
    ),
    "n_examples": int(len(pred_df)),
}
metrics

## Error analysis

In [ ]:
errors = pred_df[
    (pred_df["predicted_jurisdiction_level"] != pred_df["expected_jurisdiction_level"]) |
    (pred_df["predicted_responsible_body"] != pred_df["expected_responsible_body"])
]
errors

In [ ]:
(ARTIFACTS_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
pred_df.to_json(ARTIFACTS_DIR / "eval_predictions.json", orient="records", indent=2)

print("Saved metrics and predictions to:", ARTIFACTS_DIR.resolve())

## DVC scaffold

In [ ]:
print((ROOT / "dvc.yaml").read_text())

In [ ]:
print((ROOT / "params.yaml").read_text())

## Suggested terminal workflow

Run these commands from the project root after installing DVC:

```bash
dvc repro
dvc metrics show
```

## Exercise

1. Add five harder evaluation questions.
2. Compare a keyword baseline, a retrieval-augmented model call, and a tool-using agent.
3. Track the predictions and metrics in the `artifacts/` directory.
4. Update the DVC stages if you add a new retrieval or indexing step.

## Reflection

An agent demo becomes an ML system only when it is grounded, testable, reproducible, and measurable.